In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# Φόρτωση του feature extractor του βασικού μοντέλου
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/data2vec-audio-base-960h")

In [ ]:
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

dataset = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print("Πριν το split:", dataset.column_names)  # έλεγχος 1

splitter = GroupShuffleSplit(test_size=0.35, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

print("Μετά το split:", dataset["train"].column_names)  # έλεγχος 2
print(dataset["train"][0]["labels"])  # έλεγχος 3 — πρέπει να δώσει int (π.χ. 0-7)

In [ ]:
dataset_raw = load_dataset("narad/ravdess", revision="refs/convert/parquet")["train"]

label_names = dataset_raw.features["labels"].int2str(list(range(dataset_raw.features["labels"].num_classes)))
num_labels = dataset_raw.features["labels"].num_classes
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}
print(label_names)

In [ ]:
columns_to_remove = [c for c in dataset["train"].column_names if c != "labels"]
print(columns_to_remove)  # πρέπει: ['audio', 'text', 'speaker_id', 'speaker_gender']

def prepare_dataset(batch):
    audio = batch["audio"]
    max_samples = 6 * 16000
    audio_array = audio["array"][:max_samples]
    batch["input_values"] = feature_extractor(audio_array, sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = int(batch["labels"])
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=columns_to_remove)

In [ ]:
from collections import Counter
print(Counter(dataset["train"]["labels"] if "labels" in dataset["train"].column_names else dataset["train"]["emotion"]))

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
import numpy as np

#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

#Συνάρτηση υπολογισμού των μετρικών αξιολόγησης
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    roc_auc = roc_auc_score(
        labels_bin, probs, multi_class="ovr", average="macro"
    )

    # PR-AUC (macro-averaged average precision)
    pr_auc = average_precision_score(
        labels_bin, probs, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted")
    f1 = f1_score(labels, predictions, average="weighted")
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs)

    cm = confusion_matrix(labels, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
#Φόρτωση του αρχικού μοντελου και εισαγωγή των ετικετών
model = AutoModelForAudioClassification.from_pretrained("facebook/data2vec-audio-base-960h",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

model.freeze_feature_encoder()

#Φόρτωση μοντέλου στην GPU
if torch.cuda.is_available():
    model = model.to('cuda')
print(f"Το μοντέλο φορτώθηκε στη συσκευή: {model.device}")

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
# Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 15

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(encoded_dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))

print(f"Steps/epoch: {steps_per_epoch}, Total steps: {total_steps}, Warmup steps: {warmup_steps}")

#Καθορισμός μετρικών εκπαίδευσης του μοντέλου
training_args = TrainingArguments(
    output_dir="./data2vec-finetuned-ravdess",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=True,
    logging_steps=50,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)


#Δημιουργία DataCollator για padding των δεδομένων
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor([feature["labels"] for feature in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = encoded_dataset["train"]["labels"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to("cuda")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import EarlyStoppingCallback

#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-ravdess"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")